# Cuaderno final del PFI: score de pago tardío para la cartera

**Proyecto Final Integrador · Machine Learning I · Universidad EAN** · Equipo **ArbolesDeIndecision**
Integrantes: Claudia Naranjo (Jefa), Yery Pedraza (Líder) y Nicolás Ruiz (Crack).

Recorre, de principio a fin, la cadena que sostiene `informe_PFI.pdf`: datos y etiqueta, partición,
comparación de modelos, modelo final, evaluación en la prueba, backtest, y los componentes A y C.
El código común vive en `src/cartera/`; cada sección remite al cuaderno del reto que tiene el
detalle. Cada sección termina con una **verificación**: la cifra calculada aquí debe coincidir con
la del informe.

| Sección | Qué hace | Detalle en |
|---|---|---|
| §1 Datos y etiqueta | Carga, construye la etiqueta y cuenta las facturas | `retos/modulo-2b/diagnostico_R11.ipynb` |
| §2 Partición | Corte temporal y pliegues por trimestre, sin solapamiento | `retos/modulo-2b/preparacion_R12.ipynb` |
| §3 Comparación | Línea base y seis familias bajo la misma validación cruzada | `modelos_R13.ipynb`, `red_R14.ipynb` |
| §4 Modelo final | Logística sin pesos de clase, verificada en la validación | `entrega/evaluacion_B.ipynb` §1 |
| §5 Prueba | **Reproduce** las métricas del 21/10/2026 | `entrega/evaluacion_B.ipynb` §3 |
| §6 Backtest | Monto en riesgo capturado con K = 10, 20 y 40 | `entrega/evaluacion_B.ipynb` §3 |
| §7 A y C | Segmentos de clientes y días hasta el pago | `retos/modulo-3/`, `entrega/plazo_C.ipynb` |

**Cómo correrlo.** Desde el repositorio, con los datos de la compañía en `datos_crudos/` (no se
redistribuyen; ver `datos_crudos/LEEME.md`) y el entorno de `requirements.txt`. Usar
*Reiniciar y ejecutar todo*. Tarda unos minutos, sobre todo por la búsqueda de hiperparámetros de §3.

## 0. Configuración

In [1]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datos_crudos").is_dir())
sys.path.insert(0, str(RAIZ / "src"))

from cartera import backtest as bt
from cartera import datos
from cartera import diseno as dis
from cartera import etiqueta as et
from cartera import historial as hi
from cartera import muestra as mu
from cartera import segmentacion as sg
from cartera.config import SEMILLA
from cartera.diseno import BLOQUE, CAT, CORTE, DESDE, FECHA_CORTE, NUM

np.random.seed(SEMILLA)
print("Semilla", SEMILLA, "| Python", sys.version.split()[0], "| pandas", pd.__version__,
      "| numpy", np.__version__, "| scikit-learn", sklearn.__version__, "| scipy", scipy.__version__)

Semilla 2026 | Python 3.14.5 | pandas 3.0.6 | numpy 2.5.3 | scikit-learn 1.9.1 | scipy 1.18.1


## 1. Datos y etiqueta

ERP de la compañía, anonimizado por ella y usado con autorización escrita. La etiqueta `tardía` vale
1 si la factura se pagó más de 30 días después de vencer, midiendo el atraso **hasta el último recibo
de caja**. Las facturas cerradas sin ningún recibo y las mixtas cuyo recibo cubre menos del 90 % del
valor quedan sin etiqueta (`src/cartera/etiqueta.py`).

In [2]:
saldos, movs = datos.leer_todo()
cerradas = et.facturas_cerradas(saldos)
atraso = et.atraso_etiqueta(cerradas, movs)
etiquetadas = atraso.dropna()
tardia = et.pago_tardio(etiquetadas)

resumen_datos = pd.Series({
    "Saldos (filas × columnas)": f"{saldos.shape[0]:,} × {saldos.shape[1]}",
    "Movimientos (filas × columnas)": f"{movs.shape[0]:,} × {movs.shape[1]}",
    "Facturas cerradas con atraso medible": len(cerradas),
    "Facturas con etiqueta": len(etiquetadas),
    "Facturas tardías": int(tardia.sum()),
    "% de pago tardío": round(tardia.mean() * 100, 2),
}, dtype=object)
resumen_datos.to_frame("valor")

,valor
Saldos (filas × columnas),"19,262 × 54"
Movimientos (filas × columnas),"40,519 × 44"
Facturas cerradas con atraso medible,17209
Facturas con etiqueta,15822
Facturas tardías,5196
% de pago tardío,32.84


In [3]:
# Verificación: informe §2 (figura 2)
assert saldos.shape == (19262, 54) and movs.shape == (40519, 44)
assert len(etiquetadas) == 15822 and int(tardia.sum()) == 5196
print("§1 coincide con el informe")

§1 coincide con el informe


## 2. Partición temporal sin fuga

Corte el 1/07/2025: entrenamiento antes, prueba después. Solo entran facturas cuyo desenlace ya se
conocía en cada corte. Las predictoras son las 11 conocidas al emitir la factura; el historial del
cliente se calcula solo con facturas anteriores. La validación cruzada usa 5 pliegues de ventana
creciente por trimestre: siempre se entrena con el pasado.

In [4]:
conj = dis.conjuntos(saldos, movs)
subent, val, ent, prueba = (conj[n] for n in ("subentrenamiento", "validación", "entrenamiento completo", "prueba"))
X, y = ent[NUM + CAT], ent.tardia

cv = dis.pliegues(ent, n=5)
trimestre = ent.fecha.dt.to_period(BLOQUE)
filas = []
for k, (i_ent, i_val) in enumerate(cv, 1):
    assert not set(trimestre.iloc[i_ent]) & set(trimestre.iloc[i_val]), "un trimestre quedó en los dos lados"
    inicio = trimestre.iloc[i_val].iloc[0].start_time
    assert (ent.fecha_vencimiento.iloc[i_ent] + pd.Timedelta(days=30) <= inicio).all(), "desenlace no conocido"
    filas.append({"pliegue": k, "trimestre validado": str(trimestre.iloc[i_val].iloc[0]),
                  "facturas entrenamiento": len(i_ent), "facturas validación": len(i_val)})
print("Predictoras:", NUM + CAT)
print(f"Entrenamiento: {len(ent)} facturas ({y.mean() * 100:.1f} % tardías) | prueba: {len(prueba)} facturas")
pd.DataFrame(filas).set_index("pliegue")

Predictoras: ['plazo_dias_observado', 'log_monto_inicial', 'fecha_mes', 'atraso_prom_hist', 'n_pagadas_hist', 'saldo_abierto_t', 'n_abiertas_t', 'n_vencidas_t', 'antiguedad_dias', 'unidad_negocio', 'tipo_cliente']
Entrenamiento: 8930 facturas (30.2 % tardías) | prueba: 3970 facturas


,trimestre validado,facturas entrenamiento,facturas validación
pliegue,,,
1,2024Q2,5323,704
2,2024Q3,5972,749
3,2024Q4,6720,744
4,2025Q1,7477,698
5,2025Q2,8199,470


In [5]:
# Verificación: informe §3 (figura 3)
assert len(ent) == 8930 and len(prueba) == 3970
assert ent.fecha.max() < CORTE <= prueba.fecha.min()
print("§2 coincide con el informe")

§2 coincide con el informe


## 3. Comparación de modelos

Línea base y seis familias, con el mismo `Pipeline` de preparación ajustado dentro de cada pliegue y
los mismos 5 pliegues. Métrica: AUC-PR. El desbalance se trata con pesos de clase balanceados, solo
al entrenar. Los dos ensambles se ajustan con la misma búsqueda de R13: 30 combinaciones al azar por
ensamble, con esta misma validación cruzada y la misma semilla (`modelos_R13.ipynb` §2). La red usa la configuración elegida en R14 (una capa de 32 neuronas, α = 1, 100 épocas),
con 5 semillas.

**Regla fijada antes de ver resultados:** un modelo reemplaza a la regresión logística solo si su
mejora supera la variación entre pliegues.

In [6]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore", category=ConvergenceWarning)

def pipe(prep, est):
    return Pipeline([("prep", prep), ("est", est)])

CANDIDATOS = {
    "Línea base (Dummy)": pipe(dis.prep_lineal(), DummyClassifier(strategy="prior")),
    "Árbol de referencia": pipe(dis.prep_arbol(), DecisionTreeClassifier(
        max_depth=6, min_samples_leaf=50, class_weight="balanced", random_state=SEMILLA)),
    "Regresión logística": pipe(dis.prep_lineal(), LogisticRegression(class_weight="balanced", max_iter=2000)),
    "SVM RBF": pipe(dis.prep_lineal(), SVC(kernel="rbf", class_weight="balanced")),
}

# Búsqueda de hiperparámetros de los ensambles, igual que en R13
from scipy.stats import loguniform
from sklearn.model_selection import RandomizedSearchCV

ESPACIOS = {
    "Bosque aleatorio (ajustado)": (
        pipe(dis.prep_arbol(), RandomForestClassifier(class_weight="balanced", n_jobs=1, random_state=SEMILLA)),
        {"est__n_estimators": [200, 400], "est__max_depth": [None, 6, 10, 16],
         "est__min_samples_leaf": [1, 5, 20, 50], "est__max_features": ["sqrt", 0.5, None]}),
    "Gradient boosting (ajustado)": (
        pipe(dis.prep_arbol(), HistGradientBoostingClassifier(class_weight="balanced", random_state=SEMILLA)),
        {"est__learning_rate": loguniform(0.02, 0.3), "est__max_leaf_nodes": [7, 15, 31, 63],
         "est__min_samples_leaf": [20, 50, 100, 200], "est__l2_regularization": loguniform(1e-3, 10),
         "est__max_iter": [100, 200, 400]}),
}
inicio = time.perf_counter()
for nombre, (modelo, espacio) in ESPACIOS.items():
    b = RandomizedSearchCV(modelo, espacio, n_iter=30, cv=cv, scoring="average_precision",
                           random_state=SEMILLA, n_jobs=-1).fit(X, y)
    CANDIDATOS[nombre] = modelo.set_params(**b.best_params_)
    print(nombre, "→", {k.replace("est__", ""): (round(v, 4) if isinstance(v, float) else v)
                        for k, v in b.best_params_.items()})

ap = {}
for nombre, modelo in CANDIDATOS.items():
    ap[nombre] = cross_validate(modelo, X, y, cv=cv, scoring="average_precision")["test_score"]

# Red neuronal: pesos de muestra balanceados (MLPClassifier no tiene class_weight), 5 semillas
pesos = compute_sample_weight("balanced", y)
por_semilla = [cross_validate(pipe(dis.prep_lineal(), MLPClassifier(
                   hidden_layer_sizes=(32,), alpha=1.0, max_iter=100, early_stopping=False, random_state=s)),
               X, y, cv=cv, scoring="average_precision", params={"est__sample_weight": pesos})["test_score"]
               for s in range(SEMILLA, SEMILLA + 5)]
ap["Red neuronal (MLP)"] = np.mean(por_semilla, axis=0)
print(f"Validación cruzada: {time.perf_counter() - inicio:.0f} s")

comparacion = pd.DataFrame({"AUC-PR media": {n: v.mean() for n, v in ap.items()},
                            "desv. entre pliegues": {n: v.std() for n, v in ap.items()}})
comparacion.sort_values("AUC-PR media", ascending=False).round(4)

Bosque aleatorio (ajustado) → {'n_estimators': 400, 'min_samples_leaf': 5, 'max_features': 'sqrt', 'max_depth': 16}


Gradient boosting (ajustado) → {'l2_regularization': np.float64(0.0016), 'learning_rate': np.float64(0.0668), 'max_iter': 100, 'max_leaf_nodes': 15, 'min_samples_leaf': 100}


Validación cruzada: 105 s


,AUC-PR media,desv. entre pliegues
Red neuronal (MLP),0.7295,0.0346
Regresión logística,0.7203,0.0213
Gradient boosting (ajustado),0.7196,0.0433
Bosque aleatorio (ajustado),0.7193,0.0494
SVM RBF,0.7000,0.0366
Árbol de referencia,0.6653,0.0303
Línea base (Dummy),0.3095,0.0068


In [7]:
mejor = comparacion.drop(index="Línea base (Dummy)")["AUC-PR media"].idxmax()
mejora = ap[mejor].mean() - ap["Regresión logística"].mean()
ELEGIDO = mejor if (mejor != "Regresión logística" and mejora > ap[mejor].std()) else "Regresión logística"
pd.Series({"Mejor AUC-PR media": mejor, "Mejora sobre la logística": round(mejora, 3),
           "Variación entre pliegues del mejor": round(ap[mejor].std(), 3),
           "Modelo final": ELEGIDO}, dtype=object).to_frame("valor")

,valor
Mejor AUC-PR media,Red neuronal (MLP)
Mejora sobre la logística,0.009
Variación entre pliegues del mejor,0.035
Modelo final,Regresión logística


In [8]:
# Verificación: informe §4 (figura 4)
assert round(ap["Regresión logística"].mean(), 3) == 0.720
assert round(ap["Bosque aleatorio (ajustado)"].mean(), 3) == 0.719
assert round(ap["Gradient boosting (ajustado)"].mean(), 3) == 0.720
assert round(ap["Red neuronal (MLP)"].mean(), 4) == 0.7295
assert round(ap["Línea base (Dummy)"].mean(), 3) == 0.309
assert ELEGIDO == "Regresión logística"
print("§3 coincide con el informe (la red da 0,7295: 0,730 en el informe)")

§3 coincide con el informe (la red da 0,7295: 0,730 en el informe)


## 4. Modelo final: logística sin pesos de clase

La probabilidad se multiplica por el saldo, así que debe estar calibrada. Con pesos balanceados la
logística sobrestima todas las probabilidades; sin pesos, por cómo se ajusta, queda calibrada. Se
verifica en la validación (2024-07 a 2025-06), entrenando con el subentrenamiento.

In [9]:
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

def logistica(pesos=None):
    return pipe(dis.prep_lineal(), LogisticRegression(class_weight=pesos, max_iter=2000))

X_val, y_val = val[NUM + CAT], val.tardia
p_con = logistica("balanced").fit(subent[NUM + CAT], subent.tardia).predict_proba(X_val)[:, 1]
modelo_val = logistica().fit(subent[NUM + CAT], subent.tardia)
p_sin = modelo_val.predict_proba(X_val)[:, 1]
pd.DataFrame({
    "con pesos balanceados": {"AUC-PR": average_precision_score(y_val, p_con), "Brier": brier_score_loss(y_val, p_con)},
    "sin pesos (modelo final)": {"AUC-PR": average_precision_score(y_val, p_sin), "Brier": brier_score_loss(y_val, p_sin)},
}).round(3)

,con pesos balanceados,sin pesos (modelo final)
AUC-PR,0.724,0.723
Brier,0.152,0.145


In [10]:
# Verificación: informe §5
assert round(average_precision_score(y_val, p_sin), 3) == 0.723 and round(brier_score_loss(y_val, p_sin), 3) == 0.145
print("§4 coincide con el informe")

§4 coincide con el informe


## 5. Evaluación en la prueba: reproducción

**Esta sección reproduce un resultado ya obtenido; no se usa para decidir nada.** La prueba se
consultó una sola vez, el 21/10/2026 a las 22:09, en `entrega/evaluacion_B.ipynb` §3, con el modelo,
la lista y la métrica ya fijados. Aquí se repite el mismo cálculo, con la misma semilla, para que el
cuaderno corra de principio a fin; la verificación comprueba que da exactamente lo mismo. Si diera
otra cosa, el cuaderno se detiene: no se ajusta nada para que coincida.

In [11]:
modelo_final = logistica().fit(X, y)
X_pr, y_pr = prueba[NUM + CAT], prueba.tardia
p_pr = modelo_final.predict_proba(X_pr)[:, 1]

rng = np.random.default_rng(SEMILLA)
boot = []
for _ in range(1000):
    i = rng.integers(0, len(y_pr), len(y_pr))
    boot.append(average_precision_score(y_pr.to_numpy()[i], p_pr[i]))
auc_pr = average_precision_score(y_pr, p_pr)
pd.Series({"Facturas de prueba": len(prueba), "% tardías (AUC-PR de la línea base)": round(y_pr.mean() * 100, 1),
           "AUC-PR": round(auc_pr, 3), "IC 95 % (bootstrap)": f"{np.percentile(boot, 2.5):.3f} a {np.percentile(boot, 97.5):.3f}",
           "AUC-ROC": round(roc_auc_score(y_pr, p_pr), 3), "Brier": round(brier_score_loss(y_pr, p_pr), 3)},
          dtype=object).to_frame("valor")

,valor
Facturas de prueba,3970
% tardías (AUC-PR de la línea base),38.6
AUC-PR,0.744
IC 95 % (bootstrap),0.721 a 0.768
AUC-ROC,0.829
Brier,0.177


In [12]:
# Verificación: informe §5 y evaluacion_B.ipynb §3
assert round(auc_pr, 3) == 0.744 and round(roc_auc_score(y_pr, p_pr), 3) == 0.829
assert round(y_pr.mean() * 100, 1) == 38.6
print("§5 reproduce exactamente el resultado del 21/10/2026")

§5 reproduce exactamente el resultado del 21/10/2026


## 6. Backtest: monto en riesgo capturado

Cada día hábil del 1/07/2025 al 18/08/2026 se reconstruye la cartera con los movimientos anteriores a
ese día y se arman tres listas: la regla actual (saldo vencido), la segunda línea base (saldo vencido
en facturas de 1 a 30 días de mora) y el modelo (probabilidad × saldo). Métrica: del dinero en duda
que terminó tardío, qué parte estaba en la lista. Es parte de la misma reproducción de §5.

In [13]:
facturas = saldos[saldos.es_deuda_facturada & (saldos.plazo_dias_observado >= 0) & (saldos.fecha >= "2021-06-01")]
facturas = mu.agregar_log_monto_inicial(facturas, movs)
facturas = facturas.join(hi.calcular(facturas, saldos, movs))
probabilidad = pd.Series(modelo_final.predict_proba(facturas[NUM + CAT])[:, 1], index=facturas.id_saldo)

sim = bt.simular(bt.Cartera(saldos, movs), pd.bdate_range(CORTE, "2026-08-18"), probabilidad,
                 bt.desenlace_final(saldos, movs, FECHA_CORTE))
resultado = bt.resumir(sim)
resultado.iloc[:, :1].unstack("lista")

monto en riesgo capturado (%)                      \
lista                        modelo regla actual   
k                                                         
10                             32.0                24.0   
20                             47.4                37.9   
40                             67.1                57.3   

                                   
lista saldo de 1 a 30 días  
k                                  
10                           26.9  
20                           41.4  
40                           60.1

In [14]:
# Verificación: informe §5 (figura 5)
capturado = resultado["monto en riesgo capturado (%)"]
assert capturado[(20, "modelo")] == 47.4 and capturado[(20, "regla actual")] == 37.9
assert capturado[(20, "saldo de 1 a 30 días")] == 41.4
print("§6 coincide con el informe")

§6 coincide con el informe


## 7. Componentes A y C

**A · Segmentación de clientes.** K-means con k = 2 sobre siete variables del perfil de pago hasta
junio de 2025 (`retos/modulo-3/`). Define el tono del contacto; no entra al modelo B.
**C · Plazo real.** Días desde la emisión hasta el último recibo de caja, por segmento, con facturas
pagadas antes del corte (`entrega/plazo_C.ipynb`).

In [15]:
from sklearn.cluster import KMeans

perfil = sg.perfil(saldos, movs)
ent_c, prueba_c = sg.particion(perfil, SEMILLA)   # partición de clientes de A, no la prueba temporal
prep = sg.preparacion().fit(ent_c[sg.VARIABLES])
km = KMeans(sg.K, n_init=sg.N_INIT, random_state=SEMILLA).fit(prep.transform(ent_c[sg.VARIABLES]))
orden = np.argsort([np.median(ent_c.atraso_mediano.to_numpy()[km.labels_ == g]) for g in range(sg.K)])
renum = np.argsort(orden)
NOMBRES = {0: "paga a tiempo o casi", 1: "paga tarde"}
segmento = pd.concat([pd.Series(renum[km.labels_], index=ent_c.index),
                      pd.Series(renum[km.predict(prep.transform(prueba_c[sg.VARIABLES]))], index=prueba_c.index)]
                     ).map(NOMBRES).rename("segmento")

pagadas = cerradas[(cerradas.fecha >= DESDE) & (cerradas.fecha < CORTE)
                   & (cerradas.fecha_cancelacion_registrada < CORTE) & (cerradas.plazo_dias_observado >= 0)]
pagadas = pagadas.assign(atraso=et.atraso_etiqueta(pagadas, movs)).dropna(subset=["atraso"])
pagadas = pagadas.assign(dias_pago=pagadas.plazo_dias_observado + pagadas.atraso).join(segmento, on="nit_cliente", how="inner")
tabla_ac = pagadas.groupby("segmento").agg(clientes=("nit_cliente", "nunique"), facturas=("dias_pago", "size"),
                                           plazo_mediano=("plazo_dias_observado", "median"),
                                           dias_pago_mediana=("dias_pago", "median"))
tabla_ac.insert(0, "clientes segmentados", segmento.value_counts())
tabla_ac

,clientes segmentados,clientes,facturas,plazo_mediano,dias_pago_mediana
segmento,,,,,
paga a tiempo o casi,171,171,7088,15.0,28.0
paga tarde,55,55,1494,15.0,72.5


In [16]:
# Verificación: informe §4 y plazo_C.ipynb §2
assert segmento.value_counts().to_dict() == {"paga a tiempo o casi": 171, "paga tarde": 55}
assert tabla_ac.dias_pago_mediana.to_dict() == {"paga a tiempo o casi": 28.0, "paga tarde": 72.5}
print("§7 coincide con el informe")

§7 coincide con el informe


## Cierre

Todas las verificaciones pasaron: el cuaderno reproduce, desde un núcleo reiniciado y con la semilla
2026, las cifras que sostiene el informe. El detalle de cada paso, las alternativas descartadas y los
análisis complementarios (sensibilidad de la red, importancia por permutación, auditoría por grupo,
contraste de la segmentación, piloto de C) están en los cuadernos de cada reto.